In [ ]:
# ============================================================
# RAW → CLEAN (counts for multis) → TARGET CODING → SAVE
# ============================================================
import pandas as pd
import numpy as np
import re

# ---------- 0) Load ----------
in_path = "/content/survey_results_public.csv"   # change to your full file later
df = pd.read_csv(in_path)
print("Raw shape:", df.shape)

# ---------- 1) Clean column names (stable, ASCII-ish) ----------
# - strip, lower/keep case? we'll keep original case, just make them safe.
# - replace whitespace with '_', drop weird punctuation, collapse repeats
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.replace(r"[’‘´`]", "'", regex=True)                     # normalize apostrophes
      .str.replace(r"\s+", "_", regex=True)                        # spaces -> _
      .str.replace(r"[^0-9A-Za-z_]", "", regex=True)               # remove other odd chars
      .str.replace(r"__+", "_", regex=True)                        # collapse __
)

# ---------- 2) Drop obvious metadata/leakage ----------
drop_cols = [
    "ResponseId","Check","SurveyLength","SurveyEase",
    "ConvertedCompYearly","CompTotal","Country","Currency"
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True, errors="ignore")

# ---------- 3) Separate targets & AINext (for later derived JobPerspectiveClass) ----------
target_cols = [c for c in ["JobSat","Frustration"] if c in df.columns]
ainext_cols  = [c for c in df.columns if c.startswith("AINext")]
df_targets   = df[target_cols + ainext_cols].copy()
df_features  = df.drop(columns=target_cols + ainext_cols, errors="ignore").copy()

# ---------- 4) Multi-select → COUNT (no one-hot) ----------
multi_select_cols = [
    # Tech stacks
    "LanguageHaveWorkedWith","LanguageWantToWorkWith","LanguageAdmired",
    "DatabaseHaveWorkedWith","DatabaseWantToWorkWith","DatabaseAdmired",
    "PlatformHaveWorkedWith","PlatformWantToWorkWith","PlatformAdmired",
    "WebframeHaveWorkedWith","WebframeWantToWorkWith","WebframeAdmired",
    "EmbeddedHaveWorkedWith","EmbeddedWantToWorkWith","EmbeddedAdmired",
    "MiscTechHaveWorkedWith","MiscTechWantToWorkWith","MiscTechAdmired",
    "ToolsTechHaveWorkedWith","ToolsTechWantToWorkWith","ToolsTechAdmired",
    "NEWCollabToolsHaveWorkedWith","NEWCollabToolsWantToWorkWith","NEWCollabToolsAdmired",
    "OfficeStackAsyncHaveWorkedWith","OfficeStackAsyncWantToWorkWith","OfficeStackAsyncAdmired",
    "OfficeStackSyncHaveWorkedWith","OfficeStackSyncWantToWorkWith","OfficeStackSyncAdmired",
    "AISearchDevHaveWorkedWith","AISearchDevWantToWorkWith","AISearchDevAdmired",
    # Extras
    "CodingActivities","Employment","ProfessionalTech"
]

for col in multi_select_cols:
    if col in df_features.columns:
        new_col = f"{col}_Count"
        df_features[new_col] = (
            df_features[col]
              .fillna("")
              .astype(str)
              .str.strip()
              .apply(lambda s: 0 if s == "" else len(s.split(";")))
              .astype("Int64")
        )
        df_features.drop(columns=[col], inplace=True)

# ---------- 5) Coerce key ordinals/numerics ----------
# YearsCode / YearsCodePro often "Less than 1 year" / "More than 50 years"
def _coerce_years(s):
    if s is None: return np.nan
    s = str(s)
    if s.lower().startswith("less"):  return 0.5
    if s.lower().startswith("more"):  return 51
    try:
        return float(s)
    except:
        return np.nan

for c in ["YearsCode","YearsCodePro","WorkExp"]:
    if c in df_features.columns:
        if c in ["YearsCode","YearsCodePro"]:
            df_features[c] = df_features[c].apply(_coerce_years)
        else:
            df_features[c] = pd.to_numeric(df_features[c], errors="coerce")
        # keep as float for now (half years possible), you can round later before modeling
        # if you prefer ints: uncomment
        # df_features[c] = pd.to_numeric(df_features[c], errors="coerce").round().astype("Int64")

# ---------- 6) Drop feature columns with >60% missing ----------
miss_ratio = df_features.isna().mean()
high_missing = miss_ratio[miss_ratio > 0.60].index.tolist()
if high_missing:
    print(f"Dropping {len(high_missing)} high-missing feature cols")
    df_features.drop(columns=high_missing, inplace=True)

# ---------- 7) Recombine features + targets ----------
df = pd.concat([df_features, df_targets], axis=1)
print("After cleaning/encoding shape:", df.shape)

# ---------- 8) TARGET CODING ----------

# 🔍 Inspect JobSat column
if "JobSat" in df.columns:
    print("=== JobSat ===")
    print("dtype:", df["JobSat"].dtype)
    print("Non-null count:", df["JobSat"].notna().sum(), "/", len(df))
    print("\nUnique values (first 20):")
    print(df["JobSat"].dropna().unique()[:20])
    print("\nValue counts (top 10):")
    print(df["JobSat"].value_counts(dropna=False).head(10))
else:
    print("⚠️ 'JobSat' not found in dataset.")

# --- Re-bin JobSat safely (0–10 → 3 classes) ---
if "JobSat" in df.columns:
    print("Re-binning JobSat...")

    def bin_jobsat(v):
        if pd.isna(v):
            return np.nan
        try:
            v = float(v)
        except:
            return np.nan
        if v <= 4:
            return 0   # low
        elif v <= 7:
            return 1   # medium
        else:
            return 2   # high

    # Replace JobSat with rebinned version (keep NaNs)
    df["JobSat"] = df["JobSat"].apply(bin_jobsat).astype("Int64")

    print(df["JobSat"].value_counts(dropna=False))
else:
    print("⚠️ 'JobSat' column not found in dataset.")


# Frustration: may be numeric 0..5 or text Likert. Map text if needed, then collapse to 3 classes.
# Load the dataset (replace path if needed)

# --- Define frustration mapping logic ---
frustration_options = {
    'complexity of tech stack for build',
    'complexity of tech stack for deployment',
    'amount of technical debt',
    'number of software tools in use',
    'patching/updating core components',
    'maintaining security of code being produced',
    'maintaining security of systems/platforms used in work',
    'reliability of tools/systems used in work',
    'tracking my work',
    'showing my contributions',
    'none of these'
}

def frustration_count_safe(entry):
    if pd.isna(entry) or str(entry).strip() == '':
        return np.nan
    tokens = [t.strip().lower() for t in str(entry).split(';') if t.strip() != '']
    if len(tokens) == 1 and tokens[0] == 'none of these':
        return 0
    real = [t for t in tokens if t in frustration_options and t != 'none of these']
    return len(real)

# --- Apply logic directly to Frustration column ---
df['Frustration'] = df['Frustration'].apply(frustration_count_safe)
df['Frustration'] = df['Frustration'].clip(lower=0, upper=5)  # force range 0–5

print(df['Frustration'].describe())
print("✅ Frustration column converted to ordinal (0–5)")

# (Optional) sanity print
for t in [c for c in ["JobSat","Frustration"] if c in df.columns]:
    vc = df[t].value_counts(dropna=False).sort_index()
    print(f"\n{t} value_counts:\n{vc.to_string()}")

# ---------- 9) Save ----------
out_path = "/content/cleaned2024dataset.csv"
df.to_csv(out_path, index=False)
print(f"\n💾 Saved cleaned dataset to: {out_path}")


Raw shape: (65437, 114)
Dropping 1 high-missing feature cols
After cleaning/encoding shape: (65437, 105)
=== JobSat ===
dtype: float64
Non-null count: 29126 / 65437

Unique values (first 20):
[ 8.  5. 10.  6.  9.  4.  7.  3.  2.  1.  0.]

Value counts (top 10):
JobSat
NaN     36311
8.0      7509
7.0      6379
6.0      3751
9.0      3626
10.0     2251
5.0      1956
3.0      1165
4.0      1130
2.0       772
Name: count, dtype: int64
Re-binning JobSat...
JobSat
<NA>    36311
2       13386
1       12086
0        3654
Name: count, dtype: Int64
count    28251.000000
mean         2.711762
std          1.522898
min          0.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: Frustration, dtype: float64
✅ Frustration column converted to ordinal (0–5)

JobSat value_counts:
JobSat
0        3654
1       12086
2       13386
<NA>    36311

Frustration value_counts:
Frustration
0.0     2364
1.0     4211
2.0     6419
3.0     6235
4.0     4254
5.0     

In [ ]:
# ============================================================
# Load cleaned dataset + Encode MainBranch, Age, RemoteWork, EdLevel,
# LearnCode, LearnCodeOnline, TechDoc
# ============================================================

import pandas as pd
import numpy as np

# === Load cleaned dataset ===
df = pd.read_csv("/content/cleaned2024dataset.csv")   # <-- adjust path if Excel
print("Loaded shape:", df.shape)

def norm_txt(x):
    """Normalize text & strip weird encodings without needing unidecode"""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        # handle common encoding issues
        s = s.encode('latin1').decode('utf-8', 'ignore')
    except:
        pass
    return s

def count_multi(s):
    if pd.isna(s): return 0
    s = str(s).strip()
    return 0 if s == "" else len(s.split(";"))

# ---------- Normalize ----------
for c in ["MainBranch","Age","RemoteWork","EdLevel","LearnCode","LearnCodeOnline","TechDoc"]:
    if c in df.columns:
        df[c] = df[c].apply(norm_txt)

# ---------- MainBranch ----------
if "MainBranch" in df.columns:
    def map_mainbranch(s):
        if pd.isna(s): return pd.NA
        s2 = s.lower()
        if "hobby" in s2: return 0
        if "learning to code" in s2: return 1
        if "not primarily a developer" in s2: return 2
        if "developer by profession" in s2: return 3
        return pd.NA
    df["MainBranchOrd"] = df["MainBranch"].apply(map_mainbranch).astype("Int64")
    df.drop(columns=["MainBranch"], inplace=True)

# ---------- Age ----------
AGE_ORDER = [
    "Under 18 years old","18-24 years old","25-34 years old",
    "35-44 years old","45-54 years old","55-64 years old","65 years or older"
]
age_to_ord = {k:i for i,k in enumerate(AGE_ORDER)}
age_to_mid = {
    "Under 18 years old":16,"18-24 years old":21,"25-34 years old":30,
    "35-44 years old":40,"45-54 years old":50,"55-64 years old":60,"65 years or older":70
}
if "Age" in df.columns:
    df["AgeOrd"] = df["Age"].map(age_to_ord).astype("Int64")
    df["AgeYearsMid"] = df["Age"].map(age_to_mid).astype("Int64")
    df.drop(columns=["Age"], inplace=True)

# ---------- RemoteWork ----------
if "RemoteWork" in df.columns:
    def map_remote(s):
        if pd.isna(s): return pd.NA
        s2 = s.lower()
        if "remote" in s2: return 2
        if "hybrid" in s2: return 1
        if "in-person" in s2 or "on-site" in s2: return 0
        return pd.NA
    df["RemoteWorkOrd"] = df["RemoteWork"].apply(map_remote).astype("Int64")
    df.drop(columns=["RemoteWork"], inplace=True)

# ---------- EdLevel ----------
def map_edlevel(s):
    if pd.isna(s): return pd.NA
    s2 = s.lower()
    if "primary" in s2: return 0
    if "secondary" in s2 or "high school" in s2: return 1
    if "some college" in s2: return 2
    if "associate" in s2: return 3
    if "bachelor" in s2 or "b.s" in s2 or "b.eng" in s2: return 4
    if "master" in s2 or "m.s" in s2 or "m.eng" in s2 or "mba" in s2: return 5
    if "professional" in s2: return 6
    if "phd" in s2 or "doctoral" in s2: return 7
    return 8
if "EdLevel" in df.columns:
    df["EdLevelOrd"] = df["EdLevel"].apply(map_edlevel).astype("Int64")
    df.drop(columns=["EdLevel"], inplace=True)

# ---------- Multi-select counts ----------
for col in ["LearnCode","LearnCodeOnline","TechDoc"]:
    if col in df.columns:
        df[col + "_Count"] = df[col].apply(count_multi).astype("Int64")
        df.drop(columns=[col], inplace=True)

# ---------- Save processed copy ----------
df.to_csv("/content/cleaned2024dataset_processed.csv", index=False)
print("✅ Saved processed dataset:", df.shape)
print("Preview:\n", df.head())


Loaded shape: (65437, 105)
✅ Saved processed dataset: (65437, 106)
Preview:
    YearsCode  YearsCodePro                DevType OrgSize PurchaseInfluence  \
0        NaN           NaN                    NaN     NaN               NaN   
1       20.0          17.0  Developer, full-stack     NaN               NaN   
2       37.0          27.0   Developer Experience     NaN               NaN   
3        4.0           NaN  Developer, full-stack     NaN               NaN   
4        9.0           NaN  Developer, full-stack     NaN               NaN   

  BuyNewTool BuildvsBuy TechEndorse        OpSysPersonal_use  \
0        NaN        NaN         NaN                      NaN   
1        NaN        NaN         NaN            MacOS;Windows   
2        NaN        NaN         NaN                  Windows   
3        NaN        NaN         NaN                      NaN   
4        NaN        NaN         NaN  Other (please specify):   

  OpSysProfessional_use  ... AINextLess_integrated AINextMuch_l

In [ ]:
# ===========================================
# 🔍 Why are expected columns "missing"?
# ===========================================
import re, unicodedata
import difflib
import pandas as pd

# 1) List the columns you *expect* to see (raw names, pre-encoding)
expected = [
    "OpSysPersonal use","OpSysProfessional use","NEWSOSites","SOVisitFreq","SOAccount","SOPartFreq",
    "SOHow","SOComm","AISelect","AISent","AIBen","AIAcc","AIComplex",
    "AIToolCurrently Using","AIToolInterested in Using","AIToolNot interested in Using",
    "AIThreat","AIEthics","AIChallenges","RemoteWork","YearsCode","YearsCodePro",
    "DevType","Industry","TimeSearching","TimeAnswering","ProfessionalCloud","ProfessionalQuestion",
    "Frequency_1","Frequency_2","Frequency_3",
]

cols = pd.Index(df.columns)

def norm_header(s: str) -> str:
    s2 = unicodedata.normalize("NFKC", str(s))
    s2 = s2.replace("\xa0", " ")  # NBSP → space
    s2 = re.sub(r"\s+", " ", s2).strip()
    return s2

# 2) Compute normalized header forms (for comparison only)
norm_map = {c: norm_header(c) for c in cols}
inv_norm = {}
for k,v in norm_map.items():
    inv_norm.setdefault(v, []).append(k)

print(f"Total columns: {len(cols)}")

# 3) For each expected name, try to find:
#    - exact match
#    - normalized match
#    - fuzzy near-matches
#    - related columns like *_Count, *_5, *_Ord already created in an earlier step
def find_related(name):
    patt_core = re.sub(r"[^A-Za-z0-9]+", ".*", name)  # loose regex
    related = [c for c in cols if re.search(patt_core, c, flags=re.IGNORECASE)]
    suffix_related = [c for c in cols if re.search(rf"{re.escape(name)}.*(_Count|_5|_Ord|_Bin)$", c, re.IGNORECASE)]
    fuzzy = difflib.get_close_matches(name, list(cols), n=5, cutoff=0.6)
    return related, suffix_related, fuzzy

report = []
for exp in expected:
    exact = exp in cols
    norm_exact = (norm_header(exp) in inv_norm)
    related, suffix_related, fuzzy = find_related(exp)
    report.append({
        "expected": exp,
        "exact_present": exact,
        "normalized_present": norm_exact,
        "normalized_matches": inv_norm.get(norm_header(exp), []),
        "suffix_related": suffix_related[:10],
        "related_regex": related[:10],
        "fuzzy": fuzzy,
    })

# 4) Print a concise summary
for r in report:
    print("\n=== ", r["expected"], "===")
    print("• exact_present:", r["exact_present"])
    print("• normalized_present:", r["normalized_present"], "->", r["normalized_matches"])
    print("• suffix_related:", r["suffix_related"])
    print("• fuzzy:", r["fuzzy"])

# 5) Show any columns that look like already-processed versions
processed_like = [c for c in cols if re.search(r"(_Count|_5|_Ord|_Bin)$", c)]
print("\n🔎 Columns that look already processed (suffixes _Count/_5/_Ord/_Bin):")
print(sorted(processed_like)[:80])

# 6) Show a proposed *safe* header normalization (NOT applied yet)
proposed = [norm_header(c) for c in cols]
if proposed != list(cols):
    print("\n⚠️ Header normalization would change some names. Example changes:")
    for old, new in zip(list(cols)[:20], proposed[:20]):
        if old != new:
            print(f"  '{old}'  ->  '{new}'")
    print("\nTo apply safely (optional), run:\n  df.columns = [norm_header(c) for c in df.columns]")
else:
    print("\n✅ Column headers already look normalized.")


Total columns: 106

===  OpSysPersonal use ===
• exact_present: False
• normalized_present: False -> []
• suffix_related: []
• fuzzy: ['OpSysPersonal_use', 'OpSysProfessional_use']

===  OpSysProfessional use ===
• exact_present: False
• normalized_present: False -> []
• suffix_related: []
• fuzzy: ['OpSysProfessional_use', 'OpSysPersonal_use', 'ProfessionalCloud', 'ProfessionalQuestion', 'ProfessionalTech_Count']

===  NEWSOSites ===
• exact_present: True
• normalized_present: True -> ['NEWSOSites']
• suffix_related: []
• fuzzy: ['NEWSOSites']

===  SOVisitFreq ===
• exact_present: True
• normalized_present: True -> ['SOVisitFreq']
• suffix_related: []
• fuzzy: ['SOVisitFreq', 'SOPartFreq']

===  SOAccount ===
• exact_present: True
• normalized_present: True -> ['SOAccount']
• suffix_related: []
• fuzzy: ['SOAccount']

===  SOPartFreq ===
• exact_present: True
• normalized_present: True -> ['SOPartFreq']
• suffix_related: []
• fuzzy: ['SOPartFreq', 'SOVisitFreq']

===  SOHow ===
• exa

In [ ]:
import pandas as pd

# Load your processed file
df = pd.read_csv("/content/cleaned2024dataset_processed.csv")

cols = ["AISent", "AIAcc", "AIComplex"]

for c in cols:
    print(f"\n=== {c} ===")
    if c not in df.columns:
        print("⚠️ not found")
        continue
    # clean whitespace, drop NA, and sort case-insensitively
    s = df[c].astype(str).str.strip()
    s = s[~s.isin(["", "NA", "N/A", "NaN", "nan", "None", "NULL"])]
    uniques = sorted(s.unique().tolist(), key=lambda x: x.lower())
    for u in uniques:
        print(u)



=== AISent ===
Favorable
Indifferent
Unfavorable
Unsure
Very favorable
Very unfavorable

=== AIAcc ===
Highly distrust
Highly trust
Neither trust nor distrust
Somewhat distrust
Somewhat trust

=== AIComplex ===
Bad at handling complex tasks
Good, but not great at handling complex tasks
Neither good or bad at handling complex tasks
Very poor at handling complex tasks
Very well at handling complex tasks


In [ ]:
# ============================================================
# 🔎 Unique values for the specific columns used in your block
#    - Tries raw name, underscored, and processed variants
# ============================================================
import pandas as pd, numpy as np, re, difflib

# If df isn't already loaded:
# df = pd.read_csv("/content/cleaned2024dataset_processed.csv")

def _norm_val(x):
    if pd.isna(x): return pd.NA
    s = str(x).strip()
    return pd.NA if s.lower() in {"", "na", "n/a", "none", "null"} else s

def _candidates(raw: str):
    """Generate likely column name variants: exact, underscored, processed suffixes."""
    base = raw
    under = re.sub(r"\s+", "_", base)
    alias = [base, under]
    # common processed suffixes
    out = []
    for a in alias:
        out.append(a)
        out.extend([f"{a}_Count", f"{a}_5", f"{a}_Ord", f"{a}_Bin"])
    return list(dict.fromkeys(out))  # unique, keep order

def _pick_present(raw: str, cols: pd.Index):
    """Pick the first candidate present; else return None and a few fuzzy suggestions."""
    for cand in _candidates(raw):
        if cand in cols:
            return cand, None
    fuzzy = difflib.get_close_matches(raw, list(cols), n=5, cutoff=0.6)
    return None, fuzzy

# Expected (raw) names from your preprocessing block
expected = [
    "OpSysPersonal use","OpSysProfessional use","NEWSOSites","SOVisitFreq","SOAccount","SOPartFreq",
    "SOHow","SOComm","AISelect","AISent","AIBen","AIAcc","AIComplex",
    "AIToolCurrently Using","AIToolInterested in Using","AIToolNot interested in Using",
    "AIThreat","AIEthics","AIChallenges","RemoteWork","YearsCode","YearsCodePro","DevType","Industry",
    "TimeSearching","TimeAnswering","ProfessionalCloud","ProfessionalQuestion",
    "Frequency_1","Frequency_2","Frequency_3",
]

rows = []
cols = pd.Index(df.columns)

for raw in expected:
    actual, fuzzy = _pick_present(raw, cols)
    if actual is None:
        print(f"\n=== {raw} (missing) ===")
        if fuzzy:
            print("  close matches:", ", ".join(fuzzy))
        continue

    s = df[actual].map(_norm_val)
    vc = s.dropna().value_counts()

    print(f"\n=== {raw}  →  using column: {actual} ===")
    print(f"non-null: {int(s.dropna().shape[0])} | unique (non-null): {int(vc.shape[0])}")
    for v in vc.index.tolist():
        print(f" - {v}")

    total = s.dropna().shape[0] or 1
    for v, cnt in vc.items():
        rows.append({
            "expected": raw,
            "used_column": actual,
            "value": v,
            "count": int(cnt),
            "pct": round(float(cnt)/total, 4),
        })

# Save full report (optional)
report_df = pd.DataFrame(rows).sort_values(["expected", "count"], ascending=[True, False])
report_path = "/content/unique_values_selected_columns_resolved.csv"
report_df.to_csv(report_path, index=False)
print(f"\n✅ Saved unique-values report → {report_path}")


Streaming output truncated to the last 5000 lines.
 - Android;Arch;Debian;Fedora;iOS;MacOS;Other Linux-based;Windows
 - Cygwin;Solaris;Ubuntu;Windows
 - Android;Debian;Red Hat;Ubuntu;Windows;Windows Subsystem for Linux (WSL)
 - Arch;Debian;MacOS;Other Linux-based;Windows;Windows Subsystem for Linux (WSL)
 - Android;Cygwin;Debian;MacOS;Other Linux-based;Ubuntu;Windows;Windows Subsystem for Linux (WSL)
 - Other Linux-based;Red Hat;Ubuntu;Windows Subsystem for Linux (WSL);Other (please specify):
 - BSD;Debian;Windows;Other (please specify):
 - Arch;Cygwin;Debian;Windows;Windows Subsystem for Linux (WSL)
 - Android;Fedora;MacOS;Other Linux-based;Red Hat;Solaris;Ubuntu
 - BSD;Cygwin;Red Hat;Windows
 - Red Hat;Solaris;Ubuntu;Windows;Windows Subsystem for Linux (WSL)
 - BSD;Fedora;MacOS;Red Hat;Windows;Windows Subsystem for Linux (WSL)
 - Android;Haiku;iOS;iPadOS;MacOS;Other Linux-based;Ubuntu;Windows;Windows Subsystem for Linux (WSL)
 - Arch;BSD;Debian;Fedora;Ubuntu;Windows;Windows Subsystem

In [ ]:
# ============================================================
# Continue preprocessing: SO usage, AI block, OS, years, Likerts
# Input  : /content/cleaned2024dataset_processed.csv
# Output : /content/cleaned2024dataset_modelready.csv
# ============================================================
import pandas as pd
import numpy as np
import re

# ---- load ----
df = pd.read_csv("/content/cleaned2024dataset_processed.csv")
print("Loaded:", df.shape)

# ---------- helpers ----------
def norm_txt(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    try:
        s = s.encode("latin1").decode("utf-8", "ignore")
    except Exception:
        pass
    return s

def count_multi(s):
    if pd.isna(s): return 0
    s = str(s).strip()
    return 0 if s == "" else len([t for t in s.split(";") if t.strip()])

def yes_no_map(s):
    if pd.isna(s): return pd.NA
    t = str(s).strip().lower()
    if t in {"yes","y","true","1"}:  return 1
    if t in {"no","n","false","0"}:  return 0
    return pd.NA

def likert_5_map(s):
    if pd.isna(s): return pd.NA
    t = str(s).strip().lower()
    if "very unfav" in t or "strongly disagree" in t: return 1
    if "unfav" in t or ("disagree" in t and "strongly" not in t): return 2
    if "indifferent" in t or "neither" in t or "neutral" in t:   return 3
    if "fav" in t or ("agree" in t and "strongly" not in t):     return 4
    if "very fav" in t or "strongly agree" in t:                 return 5
    return pd.NA

def trust_5_map(s):
    if pd.isna(s): return pd.NA
    t = str(s).strip().lower()
    if "strongly dis" in t or "very distrust" in t or "dont trust" in t: return 1
    if "somewhat dis" in t or "distrust" in t:                            return 2
    if "neither" in t or "indifferent" in t:                              return 3
    if "somewhat trust" in t:                                             return 4
    if "strongly trust" in t or "very trust" in t:                        return 5
    return pd.NA

def freq_map(s):
    if pd.isna(s): return pd.NA
    t = str(s).strip().lower()
    order = [("less than once per month",0), ("monthly",1), ("few times per month",2),
             ("weekly",3), ("daily or almost daily",4), ("multiple times per day",5)]
    for k,v in order:
        if k in t: return v
    if "daily" in t:  return 4
    if "weekly" in t: return 3
    if "monthly" in t:return 1
    return pd.NA

def remote_map(s):
    if pd.isna(s): return pd.NA
    t = str(s).lower()
    if "remote" in t:  return 2
    if "hybrid" in t:  return 1
    if "in-person" in t or "on-site" in t or "onsite" in t: return 0
    return pd.NA

def coerce_years(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    if s.startswith("less"): return 0.5
    if s.startswith("more"): return 51
    try: return float(s)
    except: return np.nan

# ---------- normalize text on columns we’ll touch (if they exist) ----------
norm_cols = [
    "OpSysPersonal use","OpSysProfessional use","NEWSOSites","SOVisitFreq","SOAccount","SOPartFreq",
    "SOHow","SOComm","AISelect","AISent","AIBen","AIAcc","AIComplex",
    "AIToolCurrently Using","AIToolInterested in Using","AIToolNot interested in Using",
    "AIThreat","AIEthics","AIChallenges","RemoteWork","YearsCode","YearsCodePro","DevType","Industry"
]
for c in norm_cols:
    if c in df.columns:
        df[c] = df[c].apply(norm_txt)

# ---------- Convert raw OS fields to counts (if still raw names exist) ----------
for c in ["OpSysPersonal use","OpSysProfessional use"]:
    if c in df.columns:
        df[c.replace(" ", "_") + "_Count"] = df[c].apply(count_multi).astype("Int64")
        df.drop(columns=[c], inplace=True)

# ---------- If underscored multi-selects exist, convert them to *_Count ----------
underscore_multis = {
    "OpSysPersonal_use":               "OpSysPersonal_use_Count",
    "OpSysProfessional_use":           "OpSysProfessional_use_Count",
    "AIToolCurrently_Using":           "AIToolCurrently_Using_Count",
    "AIToolInterested_in_Using":       "AIToolInterested_in_Using_Count",
    "AIToolNot_interested_in_Using":   "AIToolNot_interested_in_Using_Count",
}
for raw, tgt in underscore_multis.items():
    if raw in df.columns and tgt not in df.columns:
        df[tgt] = df[raw].apply(count_multi).astype("Int64")
        df.drop(columns=[raw], inplace=True)

# ---------- Other multi-selects → counts (if raw names exist) ----------
for c in ["NEWSOSites","SOHow","SOComm","AISelect","AIBen","AIChallenges",
          "AIToolCurrently Using","AIToolInterested in Using","AIToolNot interested in Using"]:
    if c in df.columns:
        safe = c.replace(" ", "_").replace("/", "_")
        df[safe + "_Count"] = df[c].apply(count_multi).astype("Int64")
        df.drop(columns=[c], inplace=True)

# ---------- Frequencies & account (apply only if raw text columns exist) ----------
if "SOVisitFreq" in df.columns:
    df["SOVisitFreqOrd"] = df["SOVisitFreq"].apply(freq_map).astype("Int64")
    df.drop(columns=["SOVisitFreq"], inplace=True)
if "SOPartFreq" in df.columns:
    df["SOPartFreqOrd"] = df["SOPartFreq"].apply(freq_map).astype("Int64")
    df.drop(columns=["SOPartFreq"], inplace=True)
if "SOAccount" in df.columns:
    df["SOAccountBin"] = df["SOAccount"].apply(yes_no_map).astype("Int64")
    df.drop(columns=["SOAccount"], inplace=True)

# ---------- AI sentiment / accuracy / complexity (EXACT mappings from your uniques) ----------
def _clean(s):
    if pd.isna(s): return pd.NA
    t = str(s).strip().lower()
    return re.sub(r"\s+", " ", t)

aisent_map = {
    "very unfavorable": 1,
    "unfavorable": 2,
    "indifferent": 3,
    "favorable": 4,
    "very favorable": 5,
    "unsure": pd.NA,   # treat Unsure as missing
}
aiacc_map = {
    "highly distrust": 1,
    "somewhat distrust": 2,
    "neither trust nor distrust": 3,
    "somewhat trust": 4,
    "highly trust": 5,
}
aicomplex_map = {
    "very poor at handling complex tasks": 1,
    "bad at handling complex tasks": 2,
    "neither good or bad at handling complex tasks": 3,
    "good, but not great at handling complex tasks": 4,
    "very well at handling complex tasks": 5,
}

def map_with_dict(series, mapping):
    return series.map(_clean).map(mapping).astype("Int64")

if "AISent" in df.columns and "AISent_5" not in df.columns:
    df["AISent_5"] = map_with_dict(df["AISent"], aisent_map)
    df.drop(columns=["AISent"], inplace=True)

if "AIAcc" in df.columns and "AIAcc_5" not in df.columns:
    df["AIAcc_5"] = map_with_dict(df["AIAcc"], aiacc_map)
    df.drop(columns=["AIAcc"], inplace=True)

if "AIComplex" in df.columns and "AIComplex_5" not in df.columns:
    df["AIComplex_5"] = map_with_dict(df["AIComplex"], aicomplex_map)
    df.drop(columns=["AIComplex"], inplace=True)

# ---------- AIEthics (multi-select) → count if raw exists ----------
if "AIEthics" in df.columns and "AIEthics_Count" not in df.columns:
    df["AIEthics_Count"] = df["AIEthics"].apply(count_multi).astype("Int64")
    df.drop(columns=["AIEthics"], inplace=True)

# ---------- AIThreat text → 0/1/2 (No / Not sure / Yes) ----------
if "AIThreat" in df.columns and df["AIThreat"].dtype == "object":
    def map_ai_threat(v):
        if pd.isna(v): return pd.NA
        t = str(v).strip().lower()
        if t == "no": return 0
        if t in {"i'm not sure","im not sure","i am not sure","not sure"}: return 1
        if t == "yes": return 2
        return pd.NA
    df["AIThreat"] = df["AIThreat"].apply(map_ai_threat).astype("Int64")

# ---------- RemoteWork (if still present as text) ----------
if "RemoteWork" in df.columns and df["RemoteWork"].dtype == "object":
    df["RemoteWorkOrd"] = df["RemoteWork"].apply(remote_map).astype("Int64")
    df.drop(columns=["RemoteWork"], inplace=True)

# ---------- Years / experience ----------
for c in ["YearsCode","YearsCodePro"]:
    if c in df.columns:
        if not np.issubdtype(df[c].dtype, np.number):
            df[c] = df[c].apply(coerce_years)
if "WorkExp" in df.columns:
    df["WorkExp"] = pd.to_numeric(df["WorkExp"], errors="coerce")

# ---------- Knowledge_1..9 Likert to 1..5 (if text) ----------
for k in range(1, 10):
    col = f"Knowledge_{k}"
    if col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        if s.notna().mean() < 0.2:
            s = df[col].apply(likert_5_map)
        df[col] = pd.to_numeric(s, errors="coerce").clip(1, 5)

# ---------- Frequency_1..3 (only map if non-numeric) ----------
def map_weekly_freq(x):
    if pd.isna(x): return pd.NA
    s = str(x).strip().lower()
    if s in {"na", "", "n/a", "none"}: return pd.NA
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"\s+", " ", s)
    if "never" in s: return 0
    if "1-2" in s:  return 1
    if "3-5" in s:  return 2
    if "6-10" in s or "over 10" in s or "10+" in s: return 3
    if "once a week" in s or "twice a week" in s: return 1
    if "daily" in s: return 3
    m = re.findall(r"\d+", s)
    if m:
        n = max(int(v) for v in m)
        if n <= 0: return 0
        if n <= 2: return 1
        if n <= 5: return 2
        return 3
    return pd.NA

for c in ["Frequency_1", "Frequency_2", "Frequency_3"]:
    if c in df.columns and not np.issubdtype(df[c].dtype, np.number):
        df[c] = df[c].apply(map_weekly_freq).astype("Int64")

# ---------- ProfessionalCloud → ordinal category ----------
if "ProfessionalCloud" in df.columns and "ProfessionalCloud_Cat" not in df.columns:
    df["ProfessionalCloud_Cat"] = (
        df["ProfessionalCloud"].fillna("NA").replace({
            "Cloud only (single or multi-cloud)": 1,
            "Hybrid (on-prem and cloud)": 2,
            "On-prem": 0,
            "NA": np.nan
        }).astype("Int64")
    )
    # keep text if needed; else uncomment to drop:
    # df.drop(columns=["ProfessionalCloud"], inplace=True)

# ---------- ProfessionalQuestion → ordinal (0=lowest, 1=medium, 2=highest) ----------
if "ProfessionalQuestion" in df.columns and "ProfessionalQuestionOrdinal" not in df.columns:
    def map_prof_q(v):
        if pd.isna(v): return np.nan
        t = str(v).strip().lower()
        if t.startswith("traditional public search engine"): return 0
        if t.startswith("a coworker") or t.startswith("asking a coworker"): return 0
        if t.startswith("ai-powered search (free)") or t.startswith("ai-powered search (paid)") or t.startswith("ai-powered search"): return 0
        if t.startswith("slack search"): return 1
        if t.startswith("microsoft teams search"): return 1
        if t.startswith("do search of internal share drives") or "internal share drives" in t or "storage locations" in t: return 1
        if t.startswith("internal developer portal"): return 2
        if t.startswith("other"): return np.nan
        return np.nan
    df["ProfessionalQuestionOrdinal"] = df["ProfessionalQuestion"].apply(map_prof_q).astype("Int64")

# ---------- JobSatPoints_* keep numeric non-negative ----------
for i in [1,4,5,6,7,8,9,10,11]:
    col = f"JobSatPoints_{i}"
    if col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        df[col] = s.where(s.isna() | (s >= 0), np.nan)

# ---------- Save ----------
out_path = "/content/cleaned2024dataset_modelready.csv"
df.to_csv(out_path, index=False)
print("✅ Saved model-ready dataset to:", out_path)
print("Final shape:", df.shape)


Loaded: (65437, 106)


/tmp/ipython-input-4201384504.py:244: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["ProfessionalCloud"].fillna("NA").replace({


✅ Saved model-ready dataset to: /content/cleaned2024dataset_modelready.csv
Final shape: (65437, 108)


In [ ]:
# ============================================================
# Encode: DevType, TBranch, ICorPM, Industry  → compact features
# Input : /content/cleaned2024dataset_modelready.csv (change if needed)
# Output: /content/cleaned2024dataset_modelready_v2.csv
# ============================================================
import pandas as pd
import numpy as np

# --- load ---
in_path = "/content/cleaned2024dataset_modelready.csv"   # <- change if your path differs
df = pd.read_csv(in_path)
print("Loaded:", df.shape)

def norm_txt(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    try: s = s.encode("latin1").decode("utf-8", "ignore")
    except: pass
    return s

# ---------- DevType ----------
# Many datasets have DevType as a single value, some as multi-select.
# We'll (a) normalize text, (b) optionally count ';' selections, (c) map to a compact bucket code.
if "DevType" in df.columns:
    df["DevType"] = df["DevType"].apply(norm_txt)

    # (optional) if multi-selects exist, make a count (kept as an extra numeric)
    def _count_multi(s):
        if pd.isna(s) or str(s).strip()=="":
            return 0
        return len(str(s).split(";"))
    df["DevType_Count"] = df["DevType"].apply(_count_multi).astype("Int64")

    # Map text to compact role buckets
    # codes:
    # 0=student, 1=academic, 2=project_manager, 3=manager (people),
    # 4=fullstack, 5=backend, 6=frontend, 7=mobile, 8=embedded, 9=desktop_enterprise,
    # 10=data_ml, 11=devops, 12=cloud_infra, 13=sysadmin, 14=advocate/devrel, 15=other
    def map_devtype_bucket(s):
        if pd.isna(s) or str(s).strip()=="":
            return pd.NA
        s2 = s.lower()

        if "student" in s2: return 0
        if "academic" in s2 or "research" in s2: return 1
        if "project manager" in s2: return 2
        # explicit people manager will come from ICorPM, but some titles include "manager":
        if "manager" in s2 and "project" not in s2 and "engineering manager" in s2:
            return 3

        if "full-stack" in s2 or "full stack" in s2: return 4
        if "back-end" in s2 or "backend" in s2 or "back end" in s2: return 5
        if "front-end" in s2 or "frontend" in s2 or "front end" in s2: return 6
        if "mobile" in s2 or "ios" in s2 or "android" in s2: return 7
        if "embedded" in s2 or "devices" in s2: return 8
        if "desktop" in s2 or "enterprise applications" in s2: return 9
        if "data scientist" in s2 or "machine learning" in s2 or "ml" in s2:
            return 10
        if "devops" in s2 or "site reliability" in s2 or "sre" in s2:
            return 11
        if "cloud infrastructure" in s2 or "cloud" in s2:
            return 12
        if "system administrator" in s2 or "sys admin" in s2:
            return 13
        if "advocate" in s2 or "developer advocate" in s2 or "devrel" in s2:
            return 14
        return 15  # other / unspecified

    df["DevTypeBucket"] = df["DevType"].apply(map_devtype_bucket).astype("Int64")

    # Drop original long text to avoid leakage/sparsity; keep compact features
    df.drop(columns=["DevType"], inplace=True)

# ---------- TBranch (Yes/No) ----------
# Map to binary: Yes=1, No=0
if "TBranch" in df.columns:
    def map_yesno(s):
        if pd.isna(s): return pd.NA
        s2 = str(s).strip().lower()
        if s2 in {"yes","y","true","1"}:  return 1
        if s2 in {"no","n","false","0"}:  return 0
        return pd.NA
    df["TBranchBin"] = df["TBranch"].apply(map_yesno).astype("Int64")
    df.drop(columns=["TBranch"], inplace=True)

# ---------- ICorPM ----------
# "Individual contributor" → 0, "People manager" → 1
if "ICorPM" in df.columns:
    def map_ic_pm(s):
        if pd.isna(s): return pd.NA
        s2 = str(s).strip().lower()
        if "people manager" in s2: return 1
        if "individual contributor" in s2: return 0
        # unknown text → NA
        return pd.NA
    df["ICorPMBin"] = df["ICorPM"].apply(map_ic_pm).astype("Int64")
    df.drop(columns=["ICorPM"], inplace=True)

# ---------- Industry ----------
# Normalize text and collapse rare categories (freq < min_count) into "Other"
if "Industry" in df.columns:
    df["Industry"] = df["Industry"].apply(norm_txt)

    # strip trailing colons like "Other:" and whitespace variants
    df["Industry"] = df["Industry"].fillna("").str.replace(r":\s*$", "", regex=True).replace("", np.nan)

    # collapse rare levels
    min_count = 15  # tweak for your full dataset size
    vc = df["Industry"].value_counts(dropna=False)
    rare = set(vc[vc < min_count].index.dropna())
    df["IndustryClean"] = df["Industry"].apply(lambda x: "Other" if (pd.notna(x) and x in rare) else x)
    # optional: keep top categories only; convert to string (CatBoost-friendly)
    df["IndustryClean"] = df["IndustryClean"].astype("string")
    df.drop(columns=["Industry"], inplace=True)

print("After encodings:", df.shape)

# --- save ---
out_path = "/content/cleaned2024dataset_modelready_v2.csv"
df.to_csv(out_path, index=False)
print("💾 Saved:", out_path)


Loaded: (65437, 108)
After encodings: (65437, 109)
💾 Saved: /content/cleaned2024dataset_modelready_v2.csv


In [ ]:
# ============================================================
# Final touch-ups: Industry normalization + drop near-empty time cols
# Input : /content/cleaned2024dataset_modelready_v2.csv
# Output: /content/cleaned2024dataset_modelready_v3.csv
# ============================================================
import pandas as pd
import numpy as np
import re

in_path = "/content/cleaned2024dataset_modelready_v2.csv"
df = pd.read_csv(in_path)
print("Loaded:", df.shape)

def norm_txt(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    try:
        s = s.encode("latin1").decode("utf-8", "ignore")
    except Exception:
        pass
    # collapse multiple spaces
    s = re.sub(r"\s+", " ", s)
    return s

# ---------- INDUSTRY CLEANUP ----------
if "IndustryClean" in df.columns:
    # normalize again (trim, de-mojibake)
    df["IndustryClean"] = df["IndustryClean"].apply(norm_txt)

    # unify common synonyms into a few buckets
    def map_industry(s):
        if pd.isna(s): return np.nan
        s2 = s.lower()

        # canonical software bucket
        if ("software" in s2 and "development" in s2) or ("software dev" in s2):
            return "Software Development"

        # internet / telecom / IT services bucket
        if ("internet" in s2) or ("telecomm" in s2) or ("telecom" in s2) \
           or ("information services" in s2) or ("it services" in s2):
            return "Internet/Telecom/IT Services"

        # finance / banking
        if "bank" in s2 or "financial" in s2 or "fintech" in s2:
            return "Banking/Financial Services"

        # healthcare
        if "health" in s2 or "pharma" in s2 or "medical" in s2:
            return "Healthcare"

        # education
        if "education" in s2 or "higher education" in s2 or "edtech" in s2:
            return "Education"

        # media & advertising
        if "media" in s2 or "advertis" in s2:
            return "Media/Advertising"

        # insurance
        if "insurance" in s2:
            return "Insurance"

        # energy
        if "energy" in s2 or "utilities" in s2:
            return "Energy/Utilities"

        # government/public
        if "government" in s2 or "public sector" in s2:
            return "Government/Public"

        # manufacturing
        if "manufactur" in s2 or "automotive" in s2 or "industrial" in s2:
            return "Manufacturing/Industrial"

        # keep explicit "Other"
        if s2.startswith("other"):
            return "Other"

        return s  # leave as-is for now

    df["IndustryClean"] = df["IndustryClean"].apply(map_industry)

    # collapse rare categories again (everything below min_count → Other)
    min_count = 25  # tweak for your full dataset size
    vc = df["IndustryClean"].value_counts(dropna=False)
    rare = set(vc[vc < min_count].index.dropna())
    df["IndustryClean"] = df["IndustryClean"].apply(lambda x: "Other" if (pd.notna(x) and x in rare) else x).astype("string")

    print("\nIndustryClean value counts (top):")
    print(df["IndustryClean"].value_counts(dropna=False).head(15))

# ---------- TIME COLUMNS: drop if ~all blank ----------
for c in ["TimeSearching", "TimeAnswering"]:
    if c in df.columns:
        miss = df[c].isna().mean()
        if miss > 0.95:
            print(f"Dropping {c} (missing={miss:.1%})")
            df.drop(columns=[c], inplace=True)
        else:
            # keep numeric if not dropped
            df[c] = pd.to_numeric(df[c], errors="coerce")

# ---------- SAVE ----------
out_path = "/content/cleaned2024dataset_modelready_v3.csv"
df.to_csv(out_path, index=False)
print("\n✅ Saved:", out_path, "| shape:", df.shape)


Loaded: (65437, 109)

IndustryClean value counts (top):
IndustryClean
<NA>                                    36579
Software Development                    11918
Other                                    3077
Banking/Financial Services               3012
Internet/Telecom/IT Services             1629
Healthcare                               1277
Manufacturing/Industrial                 1265
Retail and Consumer Services             1264
Government/Public                         962
Media/Advertising                         894
Education                                 890
Transportation, or Supply Chain           859
Computer Systems Design and Services      844
Energy/Utilities                          578
Insurance                                 389
Name: count, dtype: Int64

✅ Saved: /content/cleaned2024dataset_modelready_v3.csv | shape: (65437, 109)


In [ ]:
# ==============================================
# 🔧 Fix: OS + AI Tool multi-select columns
# ==============================================
import pandas as pd
import numpy as np

# --- Load your latest dataset (adjust name if needed)
df = pd.read_csv("/content/cleaned2024dataset_modelready_v3.csv")  # or .xlsx if that’s what you saved
print("Loaded:", df.shape)

# --- Multi-select columns to convert into counts ---
multi_cols = [
    "OpSysPersonal_use",
    "OpSysProfessional_use",
    "AIToolCurrently_Using",
    "AIToolInterested_in_Using"
]

for col in multi_cols:
    if col in df.columns:
        new_col = col + "_Count"
        df[new_col] = df[col].fillna("").apply(
            lambda x: 0 if str(x).strip() == "" else len(str(x).split(";"))
        )
        df.drop(columns=[col], inplace=True)

print("✅ Converted multi-select columns → counts")
print("New shape:", df.shape)
print("Preview:")
print(df.filter(like="_Count").head())

# --- Save the updated version ---
out_path = "/content/cleaned2024dataset_v4.csv"
df.to_csv(out_path, index=False)
print(f"✅ Saved updated dataset to: {out_path}")


Loaded: (65437, 109)
✅ Converted multi-select columns → counts
New shape: (65437, 109)
Preview:
   LanguageHaveWorkedWith_Count  LanguageWantToWorkWith_Count  \
0                             0                             0   
1                             7                             8   
2                             1                             1   
3                            10                             7   
4                             6                             5   

   LanguageAdmired_Count  DatabaseHaveWorkedWith_Count  \
0                      0                             0   
1                      7                             3   
2                      1                             1   
3                      7                             4   
4                      5                             2   

   DatabaseWantToWorkWith_Count  DatabaseAdmired_Count  \
0                             0                      0   
1                             1                 

In [ ]:
# ==============================================
# 🧭 Derive JobPerspectiveClass (collapsed 3-class scale)
# ==============================================
import pandas as pd
import numpy as np

# --- Load cleaned dataset ---
in_path = "/content/cleaned2024dataset_v4.csv"  # update if needed
df = pd.read_csv(in_path)
print("Loaded dataset:", df.shape)

# --- Define AINEXT source columns ---
AINEXT = [
    "AINextMuch_more_integrated",
    "AINextMore_integrated",
    "AINextNo_change",
    "AINextLess_integrated",
    "AINextMuch_less_integrated",
]

# --- Collapse map (5 → 3) ---
LEVEL_COLLAPSE = {
    "AINextMuch_more_integrated": 2,  # high optimism
    "AINextMore_integrated": 2,
    "AINextNo_change": 1,             # neutral
    "AINextLess_integrated": 0,       # low optimism
    "AINextMuch_less_integrated": 0
}

# --- Helper to split multi-select text safely ---
def _split_list(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]

# --- Check missingness ---
missing_mask = df[AINEXT].isna()
print(f"❌ Rows with all 5 AINEXT missing: {missing_mask.all(axis=1).sum()}")
print(f"❌ Total missing cells across AINEXT columns: {int(missing_mask.sum().sum())}")

# --- Build collapsed JobPerspective matrix ---
observed = set()
for c in AINEXT:
    for lst in df[c].dropna().apply(_split_list):
        observed.update(lst)

if not observed:
    print("⚠️ No valid responses found for AINEXT columns. Skipping creation.")
    df["JobPerspective"] = np.nan
    df["JobPerspectiveClass"] = np.nan
else:
    techs = sorted(observed)
    A = pd.DataFrame(np.nan, index=df.index, columns=techs)

    for c in AINEXT:
        score = LEVEL_COLLAPSE[c]
        lists = df[c].fillna("").astype(str).str.split(";")
        for i, lst in lists.items():
            for raw in lst:
                raw = raw.strip()
                if not raw:
                    continue
                prev = A.at[i, raw]
                if pd.isna(prev):
                    A.at[i, raw] = score
                else:
                    A.at[i, raw] = max(prev, score)

    # Aggregate across all technologies → median (robust central tendency)
    df["JobPerspective"] = A.median(axis=1, skipna=True)

    # Round to nearest 0/1/2
    df["JobPerspectiveClass"] = df["JobPerspective"].round().astype("Int64")

print("\n✅ Created JobPerspective & JobPerspectiveClass columns")
print(df[["JobPerspective", "JobPerspectiveClass"]].head())

# --- Summary diagnostics ---
print("\nClass counts (including NaN):")
print(df["JobPerspectiveClass"].value_counts(dropna=False).sort_index())

print("\nClass proportions (excluding NaN):")
print((df["JobPerspectiveClass"].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + "%")

# ---------- IndustryClean: normalize + collapse ----------
import re

def _norm_txt(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    try:  # fix mojibake if any
        s = s.encode("latin1").decode("utf-8", "ignore")
    except Exception:
        pass
    s = re.sub(r"\s+", " ", s)
    return s or np.nan

if "IndustryClean" in df.columns:
    df["IndustryClean"] = df["IndustryClean"].apply(_norm_txt)

    # unify common synonyms → compact set of buckets
    def _map_industry(s):
        if pd.isna(s): return np.nan
        s2 = s.lower()

        # canonical buckets
        if "software" in s2 and "develop" in s2:
            return "Software Development"
        if "internet" in s2 or "telecom" in s2 or "telecomm" in s2 \
           or "information services" in s2 or "it services" in s2:
            return "Internet/Telecom/IT Services"
        if "bank" in s2 or "financial" in s2 or "fintech" in s2:
            return "Banking/Financial Services"
        if "health" in s2 or "pharma" in s2 or "medical" in s2:
            return "Healthcare"
        if "education" in s2 or "edtech" in s2:
            return "Education"
        if "media" in s2 or "advertis" in s2:
            return "Media/Advertising"
        if "insurance" in s2:
            return "Insurance"
        if "energy" in s2 or "utilities" in s2:
            return "Energy/Utilities"
        if "government" in s2 or "public sector" in s2:
            return "Government/Public"
        if "manufactur" in s2 or "automotive" in s2 or "industrial" in s2:
            return "Manufacturing/Industrial"
        if s2.startswith("other"):
            return "Other"
        return s  # leave as-is if unknown (we may collapse it next)

    df["IndustryClean"] = df["IndustryClean"].apply(_map_industry)

    # collapse rare buckets → "Other" (tune threshold for your full dataset)
    min_count = 25
    vc = df["IndustryClean"].value_counts(dropna=False)
    rare = set(vc[vc < min_count].index.dropna())
    df["IndustryClean"] = df["IndustryClean"].apply(
        lambda x: "Other" if (pd.notna(x) and x in rare) else x
    ).astype("string")

    # optional: a simple binary that often helps models
    df["IndustryIsSoftwareDev"] = (df["IndustryClean"] == "Software Development").astype("Int64")


# --- Save final dataset ---
out_path = "/content/cleaned2024dataset_v6.csv"
df.to_csv(out_path, index=False)
print(f"\n💾 Saved final dataset with JobPerspectiveClass → {out_path}")


Loaded dataset: (65437, 109)
❌ Rows with all 5 AINEXT missing: 31269
❌ Total missing cells across AINEXT columns: 273318

✅ Created JobPerspective & JobPerspectiveClass columns
   JobPerspective  JobPerspectiveClass
0             NaN                 <NA>
1             NaN                 <NA>
2             NaN                 <NA>
3             2.0                    2
4             NaN                 <NA>

Class counts (including NaN):
JobPerspectiveClass
0        1526
1        5517
2       27125
<NA>    31269
Name: count, dtype: Int64

Class proportions (excluding NaN):
JobPerspectiveClass
0     4.47%
1    16.15%
2    79.39%
Name: proportion, dtype: object

💾 Saved final dataset with JobPerspectiveClass → /content/cleaned2024dataset_v6.csv


In [ ]:
# ============================================================
# Remove redundant / source columns before modeling
# ============================================================
import pandas as pd

in_path = "/content/cleaned2024dataset_v6.csv"   # <-- adjust if your file name differs
df = pd.read_csv(in_path)
print("Loaded:", df.shape)

# 1) Remove duplicated column names (if any appeared during merges)
df = df.loc[:, ~df.columns.duplicated()]
print("After de-dup columns:", df.shape)

# 2) Build drop list
drop_exact = [
    # helpers/derivations we don't want to keep
    "JobPerspective",              # numeric median used to make the class
    "FrustrationCount",            # helper for Frustration
    "IndustryClean",               # keep only the binary flag (IndustryIsSoftwareDev)
]

# Drop ALL AINext* source columns, but KEEP the final JobPerspectiveClass
drop_ainext = [c for c in df.columns if c.startswith("AINext")]
drop_ainext = [c for c in drop_ainext if c != "JobPerspectiveClass"]  # safety

# Join the lists and keep only those that exist
to_drop = [c for c in (drop_exact + drop_ainext) if c in df.columns]

print(f"Dropping {len(to_drop)} columns:", to_drop)
df = df.drop(columns=to_drop)
print("Final shape:", df.shape)


# 4) Save model-ready CSV
out_path = "/content/finalcleaned.csv"
df.to_csv(out_path, index=False)
print(f"✅ Saved model-ready dataset → {out_path}")


Loaded: (65437, 112)
After de-dup columns: (65437, 112)
Dropping 7 columns: ['JobPerspective', 'IndustryClean', 'AINextMuch_more_integrated', 'AINextNo_change', 'AINextMore_integrated', 'AINextLess_integrated', 'AINextMuch_less_integrated']
Final shape: (65437, 105)
✅ Saved model-ready dataset → /content/finalcleaned.csv


In [ ]:
# ============================================================
# Collapse target variables into 3 ordinal classes (in-place)
# ============================================================
import pandas as pd
import numpy as np

# === Load your latest cleaned dataset ===
in_path = "/content/finalcleaned.csv"  # update path if needed
df = pd.read_csv(in_path)
print("Loaded:", df.shape)


# ---------- FrustrationOrdinal: 0–5 → 3 classes ----------
if "Frustration" in df.columns:
    def bin_frustration(v):
        if pd.isna(v): return np.nan
        v = float(v)
        if v <= 1: return 0   # low
        elif v <= 3: return 1 # medium
        else: return 2        # high
    df["Frustration"] = df["Frustration"].apply(bin_frustration).astype("Int64")

# ---------- JobPerspectiveClass: ensure integer 0–2 ----------
if "JobPerspectiveClass" in df.columns:
    df["JobPerspectiveClass"] = df["JobPerspectiveClass"].clip(0, 2).astype("Int64")

# ---------- Summary ----------
for c in ["JobSat", "Frustration", "JobPerspectiveClass"]:
    if c in df.columns:
        print(f"\n==== {c} ====")
        print(df[c].value_counts(dropna=False).sort_index())
        print((df[c].value_counts(normalize=True, dropna=True).sort_index() * 100).round(2).astype(str) + "%")

# ---------- Save the updated dataset ----------
out_path = "/content/dev2024allfeatures.csv"
df.to_csv(out_path, index=False)
print(f"\n✅ Saved updated dataset → {out_path}")


Loaded: (65437, 105)

==== JobSat ====
JobSat
0.0     3654
1.0    12086
2.0    13386
NaN    36311
Name: count, dtype: int64
JobSat
0.0    12.55%
1.0     41.5%
2.0    45.96%
Name: proportion, dtype: object

==== Frustration ====
Frustration
0        6575
1       12654
2        9022
<NA>    37186
Name: count, dtype: Int64
Frustration
0    23.27%
1    44.79%
2    31.94%
Name: proportion, dtype: object

==== JobPerspectiveClass ====
JobPerspectiveClass
0        1526
1        5517
2       27125
<NA>    31269
Name: count, dtype: Int64
JobPerspectiveClass
0     4.47%
1    16.15%
2    79.39%
Name: proportion, dtype: object

✅ Saved updated dataset → /content/dev2024allfeatures.csv


In [ ]:
# ============================================================
# 🌟 End-to-End: Switch ALL median imputations to MICE
#    - Safe for CatBoost/TabNet
#    - Restores integer/bounded categoricals with round+clip
# ============================================================
import pandas as pd
import numpy as np

# sklearn MICE
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

RANDOM_STATE = 42

# ============================================================
# 🔷 BLOCK A: Initial imputation on the raw-ish feature file
# ============================================================
# --- Load your cleaned 3-class dataset ---
in_path = "/content/dev2024allfeatures.csv"
df = pd.read_csv(in_path)
print("Loaded:", df.shape)

# --- Identify groups ---
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove targets from numeric imputation (safer)
target_cols = [c for c in ['JobSat', 'FrustrationOrdinal', 'JobPerspectiveClass'] if c in df.columns]
num_cols = [c for c in num_cols if c not in target_cols]
cat_cols = [c for c in cat_cols if c not in target_cols]

# Track integer-typed numeric columns before imputation (for restoration)
int_like_cols = [c for c in num_cols if pd.api.types.is_integer_dtype(df[c].dtype)]
print(f"Numeric cols (excl targets): {len(num_cols)}, Categorical cols (excl targets): {len(cat_cols)}")
print(f"Integer-like numeric cols to restore: {len(int_like_cols)}")

# --- Categorical → "__MISSING__" (unchanged logic) ---
for c in cat_cols:
    if df[c].isna().any() or (df[c] == '').any():
        df[c] = df[c].fillna("__MISSING__").replace('', "__MISSING__")

# --- Count-encoded or frequency columns → 0 (keep as before) ---
count_like_cols = [c for c in df.columns if "_Count" in c or c.startswith("Frequency_")]
for c in count_like_cols:
    if c in df.columns:
        df[c] = df[c].fillna(0)

# --- MICE for numeric columns that still have NaNs (robust version) ---
num_cols_with_nan = [c for c in num_cols if df[c].isna().any()]
print(f"Numeric columns with NaNs to MICE: {len(num_cols_with_nan)}")

if num_cols_with_nan:
    # 0) Clean non-finite values (inf/-inf) to NaN so imputer can handle them
    for c in num_cols:
        if np.isinf(df[c]).any():
            df[c] = df[c].replace([np.inf, -np.inf], np.nan)

    # 1) Identify fully observed numeric predictors to help impute the missing ones
    fully_observed = [c for c in num_cols if df[c].isna().sum() == 0]

    # 2) Drop numeric columns that are ALL NaN (imputer cannot learn from them).
    #    You can choose a simple fallback (e.g., fill 0) before/after if you need values.
    all_nan_cols = [c for c in num_cols_with_nan if df[c].isna().all()]
    if all_nan_cols:
        print(f"⚠️ Skipping all-NaN columns from MICE (will fill 0): {all_nan_cols}")
        for c in all_nan_cols:
            df[c] = 0  # simple fallback; adjust if you prefer another default
        num_cols_with_nan = [c for c in num_cols_with_nan if c not in all_nan_cols]

    # 3) If there are still columns to impute via MICE:
    if num_cols_with_nan:
        # Cap the number of predictors to keep the system stable
        # (use up to 50 fully observed predictors, most variable ones first)
        if fully_observed:
            var_series = df[fully_observed].var(numeric_only=True).sort_values(ascending=False)
            helper_preds = var_series.index.tolist()[:50]
        else:
            helper_preds = []

                # 4) Build the exact column set we will pass to the imputer
        X_cols = list(dict.fromkeys(num_cols_with_nan + helper_preds))  # preserve order & uniqueness

        # >>> Per-feature bounds to keep year-like columns non-negative and sane
        min_arr = np.full(len(X_cols), -np.inf, dtype=float)
        max_arr = np.full(len(X_cols),  np.inf, dtype=float)
        bounds = {"YearsCode": (0, 60), "YearsCodePro": (0, 60), "WorkExp": (0, 60)}
        for name, (lo, hi) in bounds.items():
            if name in X_cols:
                i = X_cols.index(name)
                min_arr[i] = lo
                max_arr[i] = hi

        imputer_A = IterativeImputer(
            estimator=BayesianRidge(),
            max_iter=20,
            sample_posterior=False,
            random_state=RANDOM_STATE,
            skip_complete=True,
            min_value=min_arr,   # was None
            max_value=max_arr,   # was None
        )

        X_num = df[X_cols].copy()
        X_num_imp = imputer_A.fit_transform(X_num)
        X_num_imp = pd.DataFrame(X_num_imp, columns=X_cols, index=df.index)

        # 5) Write back only the columns that had NaNs originally
        for c in num_cols_with_nan:
            df[c] = X_num_imp[c]

        # 6) Snap year-like columns to integers and clip to [0, 60]
        for c in ["YearsCode", "YearsCodePro", "WorkExp"]:
            if c in df.columns:
                df[c] = (
                    pd.to_numeric(df[c], errors="coerce")
                      .round()
                      .clip(0, 60)
                      .astype("Int64")
                )


# --- Restore integer-typed numeric columns (unchanged) ---
if 'ProfessionalCloud_Cat' in df.columns:
    df['ProfessionalCloud_Cat'] = (
        pd.to_numeric(df['ProfessionalCloud_Cat'], errors='coerce')
        .round()
        .clip(0, 2)
        .astype('Int64')
    )
    int_like_cols = [c for c in int_like_cols if c != 'ProfessionalCloud_Cat']

for c in int_like_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').round().astype('Int64')


# --- Final sanity checks ---
rem = df.isna().sum()
rem = rem[rem > 0]
print("\nRemaining NaN counts after BLOCK A:")
print(rem if not rem.empty else "None 🎉")

# --- Save BLOCK A output ---
out_path_A = "/content/cleaned2024dataset_imputed.csv"
df.to_csv(out_path_A, index=False)
print(f"\n✅ Saved dataset after BLOCK A (MICE) → {out_path_A}")

# ============================================================
# 🔷 BLOCK B: Reload → encode five columns → MICE those + targets
# ============================================================
# 🔹 Load your final cleaned dataset (from Block A)
df = pd.read_csv(out_path_A)
print("\nLoaded for BLOCK B:", df.shape)

# 🔹 Columns you mentioned
cols_to_check = ["OrgSize", "PurchaseInfluence", "BuyNewTool", "BuildvsBuy", "TechEndorse"]

# 🔹 Display unique values per column (sorted, cleanly formatted)
for c in cols_to_check:
    if c in df.columns:
        print(f"\n=== {c} ===")
        vals = df[c].astype(str).unique().tolist()
        print(f"Total unique: {len(vals)}")
        print(pd.Series(vals).sort_values().head(50).to_list())
    else:
        print(f"\n⚠️ Column missing: {c}")

# 0) Normalize explicit missing token to NaN (for mapping)
for c in ["OrgSize","PurchaseInfluence","BuyNewTool","BuildvsBuy","TechEndorse"]:
    if c in df.columns:
        df[c] = df[c].replace({"__MISSING__": np.nan})

# 1) OrgSize → ordinal
orgsize_map = {
    "Just me - I am a freelancer, sole proprietor, etc.": 0,
    "2 to 9 employees": 1,
    "10 to 19 employees": 2,
    "20 to 99 employees": 3,
    "100 to 499 employees": 4,
    "500 to 999 employees": 5,
    "1,000 to 4,999 employees": 6,
    "5,000 to 9,999 employees": 7,
    "10,000 or more employees": 8,
    "I don’t know": np.nan,
    "I donâ€™t know": np.nan,
}
if "OrgSize" in df.columns:
    df["OrgSizeOrd"] = df["OrgSize"].map(orgsize_map).astype("float")

# 2) PurchaseInfluence → ordinal
pi_map = {
    "I have little or no influence": 0,
    "I have some influence": 1,
    "I have a great deal of influence": 2,
}
if "PurchaseInfluence" in df.columns:
    df["PurchaseInfluenceOrd"] = df["PurchaseInfluence"].map(pi_map).astype("float")

# 3) BuildvsBuy → ordinal
bvb_map = {
    "Is set up to be customized and needs to be engineered into a usable product": 0,
    "Is ready-to-go but also customizable for growth and targeted use cases": 1,
    "Out-of-the-box is ready to go with little need for customization": 2,
}
if "BuildvsBuy" in df.columns:
    df["BuildvsBuyOrd"] = df["BuildvsBuy"].map(bvb_map).astype("float")

# 4) Multi-selects → counts
def count_multiselect(x):
    if pd.isna(x) or str(x).strip() == "":
        return 0
    return len([t for t in str(x).split(";") if t.strip()])

if "BuyNewTool" in df.columns:
    df["BuyNewTool_Count"] = df["BuyNewTool"].apply(count_multiselect).astype("int64")

if "TechEndorse" in df.columns:
    df["TechEndorse_Count"] = df["TechEndorse"].apply(count_multiselect).astype("int64")

# Ensure counts are 0 where missing
for c in ["BuyNewTool_Count","TechEndorse_Count"]:
    if c in df.columns:
        df[c] = df[c].fillna(0).astype("int64")

# ------------------------------------------------------------
# 5) 🔁 MICE for the new ordinal columns (replacing median)
#    We include count columns as fully observed predictors.
# ------------------------------------------------------------
ord_cols = [c for c in ["OrgSizeOrd","PurchaseInfluenceOrd","BuildvsBuyOrd"] if c in df.columns]
predictor_cols = [c for c in ["BuyNewTool_Count","TechEndorse_Count"] if c in df.columns]
mice_matrix_cols = ord_cols + predictor_cols

if ord_cols:
    print(f"\nMICE on ordinal columns: {ord_cols} with predictors: {predictor_cols}")
    imputer_B1 = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=20,
        sample_posterior=False,
        random_state=RANDOM_STATE,
        skip_complete=True
    )
    X = df[mice_matrix_cols].copy()
    X_imp = imputer_B1.fit_transform(X)
    X_imp = pd.DataFrame(X_imp, columns=mice_matrix_cols, index=df.index)
    # write back ONLY the ordinal cols (not overwriting counts)
    for c in ord_cols:
        df[c] = X_imp[c].round().clip(0, 8 if c=="OrgSizeOrd" else 2).astype("Int64")

# 6) Drop original verbose text columns
drop_raw = [c for c in ["OrgSize","PurchaseInfluence","BuildvsBuy","BuyNewTool","TechEndorse"] if c in df.columns]
if drop_raw:
    df.drop(columns=drop_raw, inplace=True)

# 7) Quick check
keep_cols = [c for c in ["OrgSizeOrd","PurchaseInfluenceOrd","BuildvsBuyOrd","BuyNewTool_Count","TechEndorse_Count"] if c in df.columns]
print("\n✅ Encoded+MICE-imputed columns:")
for c in keep_cols:
    print(c, "-> dtype:", df[c].dtype, "| example:", df[c].head(3).to_list())

# ------------------------------------------------------------
# 8) 🔁 MICE for targets (replacing median fills)
#    Targets are 3-class labels (0,1,2) → impute, then round+clip
#    Use new ordinals + counts as predictors.
# ------------------------------------------------------------
targets = [c for c in ["JobPerspectiveClass","JobSat","FrustrationOrdinal"] if c in df.columns]
# Make sure targets are numeric
for t in targets:
    df[t] = pd.to_numeric(df[t], errors="coerce")

if targets:
    print("\nMICE on targets (with ordinals+counts as predictors):", targets)
    pred_cols = keep_cols  # use the engineered columns as predictors
    mice_cols_targets = targets + pred_cols

    imputer_B2 = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=20,
        sample_posterior=False,
        random_state=RANDOM_STATE,
        skip_complete=True
    )
    XT = df[mice_cols_targets].copy()
    XT_imp = imputer_B2.fit_transform(XT)
    XT_imp = pd.DataFrame(XT_imp, columns=mice_cols_targets, index=df.index)

    # write back targets only, with rounding/clipping to 0..2
    for t in targets:
        before = df[t].isna().sum()
        df[t] = XT_imp[t].round().clip(0, 2).astype("Int64")
        after = df[t].isna().sum()
        print(f"{t}: NaN before {before} → after {after}")

# ------------------------------------------------------------
# 9) Optional cleanup & save
# ------------------------------------------------------------
# Drop known raw columns if present (keeps your original intent)
for col in ["ProfessionalCloud","ProfessionalQuestion"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# Final save
out_path_B = "/content/final_cleaned_dataset_2024.csv"
df.to_csv(out_path_B, index=False)

print("\n✅ Saved final dataset successfully!")
print(f"File path: {out_path_B}")
print("Shape:", df.shape)
print("Columns:", len(df.columns))
print("\nPreview:")
try:
    from IPython.display import display
    display(df.head())
except Exception:
    print(df.head())

# If running in Colab:
try:
    from google.colab import files
    files.download(out_path_B)
except Exception:
    pass

Loaded: (65437, 105)
Numeric cols (excl targets): 96, Categorical cols (excl targets): 7
Integer-like numeric cols to restore: 52
Numeric columns with NaNs to MICE: 41
⚠️ Skipping all-NaN columns from MICE (will fill 0): ['TimeSearching', 'TimeAnswering']


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(



Remaining NaN counts after BLOCK A:
JobSat                 36311
JobPerspectiveClass    31269
dtype: int64

✅ Saved dataset after BLOCK A (MICE) → /content/cleaned2024dataset_imputed.csv

Loaded for BLOCK B: (65437, 105)

=== OrgSize ===
Total unique: 11
['1,000 to 4,999 employees', '10 to 19 employees', '10,000 or more employees', '100 to 499 employees', '2 to 9 employees', '20 to 99 employees', '5,000 to 9,999 employees', '500 to 999 employees', 'I don’t know', 'Just me - I am a freelancer, sole proprietor, etc.', '__MISSING__']

=== PurchaseInfluence ===
Total unique: 4
['I have a great deal of influence', 'I have little or no influence', 'I have some influence', '__MISSING__']

=== BuyNewTool ===
Total unique: 216
['Ask a generative AI tool', 'Ask a generative AI tool;Other (please specify):', 'Ask a generative AI tool;Read ratings or reviews on third party sites like G2 Crowd', 'Ask a generative AI tool;Read ratings or reviews on third party sites like G2 Crowd;Other (please spec

,YearsCode,YearsCodePro,AIThreat,WorkExp,Knowledge_1,Knowledge_2,Knowledge_3,Knowledge_4,Knowledge_5,Knowledge_6,...,DevTypeBucket,TBranchBin,ICorPMBin,JobPerspectiveClass,IndustryIsSoftwareDev,OrgSizeOrd,PurchaseInfluenceOrd,BuildvsBuyOrd,BuyNewTool_Count,TechEndorse_Count
0,10,8,0.390951,9,2.823414,2.073360,2.972236,3.436853,3.694471,2.522189,...,3.892286,0.000000,0.017531,2,0.573137,4,1,1,0,0
1,20,17,0.149979,17,4.000000,2.000000,4.000000,4.000000,4.000000,2.000000,...,4.000000,1.000000,0.000000,2,0.598111,4,1,1,0,0
2,37,27,0.206436,29,3.518234,2.386470,3.108198,3.120597,3.455187,2.747058,...,15.000000,0.000000,0.143730,2,0.463910,4,1,1,0,0
3,4,0,0.000000,3,2.857008,2.114275,3.048013,3.566575,3.814344,2.597375,...,4.000000,0.202223,0.066501,2,0.367031,4,1,1,0,0
4,9,4,0.297802,5,3.120115,2.218214,2.937531,2.770148,3.287558,2.389024,...,4.000000,0.338073,-0.039760,2,0.535390,4,1,1,0,0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# 🔧 Snap-back (targets bounded; others just rounded) + SAVE + DOWNLOAD
# ============================================================
import re, os
import numpy as np
import pandas as pd
from datetime import datetime

def to_int(cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").round().astype("Int64")

# ---- 1) Targets ONLY → clip to 0..2
targets = [c for c in ["JobPerspectiveClass","JobSat","FrustrationOrdinal","Frustration"] if c in df.columns]
for t in targets:
    df[t] = (
        pd.to_numeric(df[t], errors="coerce")
          .round()
          .clip(0, 2)
          .astype("Int64")
    )

# ---- 2) Ordinal-like names → just round (no bounds)
ordinal_like = [
    c for c in df.columns
    if (c.endswith("Ord") or c.lower().endswith("_ordinal")) and c not in targets
]
to_int(ordinal_like)

# ---- 3) Years/experience & ages → just round
to_int([c for c in ["YearsCode","YearsCodePro","WorkExp","AgeYearsMid"] if c in df.columns])

# ---- 4) Likert blocks → just round
knowledge_cols = [c for c in df.columns if re.fullmatch(r"Knowledge_\d+", c)]
freq_cols      = [c for c in df.columns if re.fullmatch(r"Frequency_\d+", c)]
to_int(knowledge_cols + freq_cols)

# ---- 5) *_Count columns → just round
count_cols = [c for c in df.columns if c.endswith("_Count")]
to_int(count_cols)

# ---- 6) JobSatPoints_* : >0 → 1 (binary), keep NaN
for i in [1, 4, 5, 6, 7, 8, 9, 10, 11]:
    col = f"JobSatPoints_{i}"
    if col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        df[col] = s.apply(lambda x: 1 if pd.notna(x) and x > 0 else (0 if pd.notna(x) else np.nan)).astype("Int64")

# ---- Sanity check: show any still-floats among intended ints
intish_patterns = knowledge_cols + freq_cols + ordinal_like + count_cols + ["YearsCode","YearsCodePro","WorkExp","AgeYearsMid"] + targets
still_float = [c for c in intish_patterns if c in df.columns and str(df[c].dtype) == "float64"]
print("Columns still float64 (should be Int64):", still_float)

# ---- SAVE with fresh timestamp (avoid browser caching)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"/content/final_cleaned_dataset_2024_ordinal_{stamp}.csv"
pq_path  = f"/content/final_cleaned_dataset_2024_ordinal_{stamp}.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(pq_path, index=False)

print("\n✅ Saved files:")
print("CSV    :", csv_path)
print("Parquet:", pq_path)
print("Shape  :", df.shape)

# ---- DOWNLOAD CSV (Colab)
try:
    from google.colab import files
    files.download(csv_path)
except Exception:
    pass


Columns still float64 (should be Int64): []

✅ Saved files:
CSV    : /content/final_cleaned_dataset_2024_ordinal_20260212_125451.csv
Parquet: /content/final_cleaned_dataset_2024_ordinal_20260212_125451.parquet
Shape  : (65437, 103)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ------------------------------------------------------------
# 🔧 Patch: force-round remaining int-like columns that aren't "*Ord"
#     (no bounds; just round → Int64)
# ------------------------------------------------------------
import re

def to_int(cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").round().astype("Int64")

# 1) Time variables that came out floaty
to_int(["TimeSearching", "TimeAnswering"])

# 2) A* 1–5 scales (no bounds per your preference, just round)
to_int(["AISent_5", "AIAcc_5", "AIComplex_5", "AIEthics_5"])

# 3) Any Knowledge_* and Frequency_* not already Int64
knowledge_cols = [c for c in df.columns if re.fullmatch(r"Knowledge_\d+", c)]
freq_cols      = [c for c in df.columns if re.fullmatch(r"Frequency_\d+", c)]
to_int(knowledge_cols + freq_cols)

# 4) Years/experience & age midpoint (round only)
to_int([c for c in ["YearsCode", "YearsCodePro", "WorkExp", "AgeYearsMid"] if c in df.columns])

# 5) All *_Count columns (round only)
count_cols = [c for c in df.columns if c.endswith("_Count")]
to_int(count_cols)

# 6) Quick audit: which of the listed columns are still float64?
listed = [
    "YearsCode","YearsCodePro","WorkExp","TimeSearching","TimeAnswering",
    *knowledge_cols, *freq_cols,
    "AISent_5","AIAcc_5","AIComplex_5","AIEthics_5",
    # add any other specific names you care about here
]
still_float = [c for c in listed if c in df.columns and str(df[c].dtype) == "float64"]
print("🧪 Columns still float64 (should be Int64):", still_float[:15], "… total:", len(still_float))


🧪 Columns still float64 (should be Int64): [] … total: 0


In [ ]:
# --- Hard snap specific columns you listed ---
import pandas as pd
import numpy as np

def coerce_round(col, lo=None, hi=None):
    if col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce").round()
        if lo is not None or hi is not None:
            if lo is None: lo = -np.inf
            if hi is None: hi =  np.inf
            s = s.clip(lo, hi)
        df[col] = s.astype("Int64")

# Targets / small ordinals
coerce_round("ProfessionalQuestionOrdinal", 0, 2)
coerce_round("DevTypeBucket", 0, None)     # non-negative integer bucket
coerce_round("DevType_Count", 0, None)     # non-negative count

# Binary flags (clip to 0..1)
for b in ["TBranchBin", "ICorPMBin", "AIThreatBin", "IndustryIsSoftwareDev", "SOAccountBin"]:
    if b in df.columns:
        coerce_round(b, 0, 1)

# Quick audit for these exact columns
cols_check = ["ProfessionalQuestionOrdinal","DevType_Count","DevTypeBucket","TBranchBin","ICorPMBin","AIThreatBin","IndustryIsSoftwareDev"]
print({c: str(df[c].dtype) for c in cols_check if c in df.columns})
for c in cols_check:
    if c in df.columns:
        print(c, "→", df[c].dropna().head(5).to_list())


{'ProfessionalQuestionOrdinal': 'Int64', 'DevType_Count': 'Int64', 'DevTypeBucket': 'Int64', 'TBranchBin': 'Int64', 'ICorPMBin': 'Int64', 'IndustryIsSoftwareDev': 'Int64'}
ProfessionalQuestionOrdinal → [0, 0, 0, 0, 0]
DevType_Count → [0, 1, 1, 1, 1]
DevTypeBucket → [4, 4, 15, 4, 4]
TBranchBin → [0, 1, 0, 0, 0]
ICorPMBin → [0, 0, 0, 0, 0]
IndustryIsSoftwareDev → [1, 1, 0, 0, 1]


In [ ]:
# --- Snap AIThreat to integers (0..2), keep NaN ---
if "AIThreat" in df.columns:
    df["AIThreat"] = (
        pd.to_numeric(df["AIThreat"], errors="coerce")
          .round()           # 0.39→0, 0.51→1, 1.6→2, etc.
          .clip(0, 2)        # enforce valid range
          .astype("Int64")   # nullable integer dtype, preserves NaN
    )




In [ ]:
# Save and download the DataFrame (CSV)
csv_path = "/content/cleaned_dataset_2024_MICE.csv"
df.to_csv(csv_path, index=False)

from google.colab import files
files.download(csv_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ================================
# 📦 STEP 1: Import + Load Dataset
# ================================
import pandas as pd

in_path = "cleaned_dataset_2024_MICE.csv"
df = pd.read_csv(in_path)
print("✅ Original shape:", df.shape)

# ================================
# 🧹 STEP 2: Define Columns to Drop (fixed commas ✅)
# ================================
cols_to_drop = [
    "Knowledge_9",
    "DatabaseWantToWorkWith_Count",
    "PlatformWantToWorkWith_Count",
    "LanguageWantToWorkWith_Count",
    "WebframeWantToWorkWith_Count",
    "EmbeddedWantToWorkWith_Count",
    "MiscTechWantToWorkWith_Count",
    "TechEndorse_Count",
    "AIEthics_Count",
    "ToolsTechWantToWorkWith_Count",
    "NEWCollabToolsWantToWorkWith_Count",
    "OfficeStackAsyncWantToWorkWith_Count",
    "OfficeStackSyncWantToWorkWith_Count",
    "AISearchDevWantToWorkWith_Count",
    "TBranchBin",
    "PurchaseInfluenceOrd",
    "BuyNewTool_Count",
    "BuildvsBuyOrd",
    "AISent_5",
    "AIBen_Count",
    "AIAcc_5",
    "AIComplex_5",
    "AIThreat"
]

# ================================
# 🔍 STEP 3: Check which columns exist
# ================================
existing_cols = [c for c in cols_to_drop if c in df.columns]
missing_cols = [c for c in cols_to_drop if c not in df.columns]

print("\n📌 Columns FOUND in dataset:")
print(existing_cols if existing_cols else "None found")

print("\n🚫 Columns NOT found in dataset:")
print(missing_cols if missing_cols else "All found")

# ================================
# 🧽 STEP 4: Drop existing columns only
# ================================
df = df.drop(columns=existing_cols, errors='ignore')
print(f"\n✅ New shape after dropping ({len(existing_cols)} columns):", df.shape)

# ================================
# 🏷️ STEP 4.5: Rename Variables (LOCKED SCHEMA)
# ================================

rename_map_2024 = {

    # ---- Already Defined ----
    "ProfessionalTech_Count": "tech_count",
    "AIChallenges_Count": "ai_challenges",
    "LearnCode_Count": "learning_count",
    "SOHow_Count": "so_usage",
    "CodingActivities_Count": "coding_count",
    "AIToolCurrently_Using_Count": "ai_tools",

    # ---- Role / Org ----
    "OrgSizeOrd": "org_size",
    "DevType_Count": "role_count",
    "DevTypeBucket": "developer_role",
    "Employment_Count": "employment",
    "MainBranchOrd": "developer_status",
    "YearsCode": "coding_years",
    "EdLevelOrd": "education_level",

    # ---- Knowledge / Workplace Dynamics ----
    "Knowledge_1": "cross_team_interaction",
    "Knowledge_2": "knowledge_silos",
    "Knowledge_3": "info_accessibility",
    "Knowledge_4": "answer_findability",
    "Knowledge_5": "resource_awareness",
    "Knowledge_6": "repeated_questions",
    "Knowledge_7": "workflow_interruptions",
    "Knowledge_8": "tool_adequacy",
    "Knowledge_9": "tool_reimbursement",

    # ---- Tools / Tech Exposure ----
    "ToolsTechHaveWorkedWith_Count": "tools_used",
    "LanguageHaveWorkedWith_Count": "languages_used",
    "DatabaseHaveWorkedWith_Count": "databases_used",
    "EmbeddedHaveWorkedWith_Count": "embedded_used",
    "NEWCollabToolsHaveWorkedWith_Count": "dev_environments_used",
    "OfficeStackSyncHaveWorkedWith_Count": "comm_tools_used",
    "OpSysProfessional_use_Count": "work_os_used",
    "OpSysPersonal_use_Count": "personal_os_used",

    # ---- Admired Technologies ----
    "LanguageAdmired_Count": "languages_admired",
    "NEWCollabToolsAdmired_Count": "dev_env_admired",
    "OfficeStackSyncAdmired_Count": "comm_tools_admired",

    # ---- Learning / Information Sources ----
    "NEWSOSites_Count": "so_sites",
    "TechDoc_Count": "tech_docs",
    "LearnCodeOnline_Count": "learning_sources",
    "SOVisitFreqOrd": "so_frequency",
    "SOPartFreqOrd": "so_participation",

    # ---- AI ----
    "AISlect_Count": "ai_scope",

    # ---- Job Satisfaction Drivers ----
    "JobSatPoints_1": "strategy_influence",
    "JobSatPoints_7": "learning_opportunities",
    "JobSatPoints_10": "hardware_quality",
    "JobSatPoints_11": "support_network"
}

df.rename(columns=rename_map_2024, inplace=True)

print(f"\n✅ Variables renamed successfully. Total columns now: {df.shape[1]}")
# ================================
# 💾 STEP 5: Save Modified File
# ================================
out_path = "devx_2024_modelling_Dataset.csv"
df.to_csv(out_path, index=False)
print(f"📁 File saved as: {out_path}")


✅ Original shape: (65437, 103)

📌 Columns FOUND in dataset:
['Knowledge_9', 'DatabaseWantToWorkWith_Count', 'PlatformWantToWorkWith_Count', 'LanguageWantToWorkWith_Count', 'WebframeWantToWorkWith_Count', 'EmbeddedWantToWorkWith_Count', 'MiscTechWantToWorkWith_Count', 'TechEndorse_Count', 'AIEthics_Count', 'ToolsTechWantToWorkWith_Count', 'NEWCollabToolsWantToWorkWith_Count', 'OfficeStackAsyncWantToWorkWith_Count', 'OfficeStackSyncWantToWorkWith_Count', 'AISearchDevWantToWorkWith_Count', 'TBranchBin', 'PurchaseInfluenceOrd', 'BuyNewTool_Count', 'BuildvsBuyOrd', 'AISent_5', 'AIBen_Count', 'AIAcc_5', 'AIComplex_5', 'AIThreat']

🚫 Columns NOT found in dataset:
All found

✅ New shape after dropping (23 columns): (65437, 80)

✅ Variables renamed successfully. Total columns now: 80
📁 File saved as: devx_2024_modelling_Dataset.csv
